# SYNCED: Production MLP Validation Notebook

**Status:** This notebook is now aligned with the FYP production pipeline.

Visible sync checks:

- Final production model: **Multilayer Perceptron** / `MLPClassifier`
- Classifier features: only 5 raw frontend/backend fields
- Blocked from classifier: `monthly_expense_total`, `surplus`, `expense_ratio`, `debt_pressure`, labels, cluster IDs, and direct leaky columns
- Holdout split: stratified from deployable `monthly_income`, matching `src/ml_pipeline.py`
- MLP params: `hidden_layer_sizes=(64, 32)`, `alpha=0.001`, `early_stopping=True`
- Random Forest remains in the comparison as a baseline candidate
- Model comparison includes Logistic Regression, Decision Tree, Multilayer Perceptron, and Random Forest

If you still see the old notebook text where Logistic Regression is selected, close the notebook tab and reopen this file from disk. A fresh copy is also saved as `clean_finance_risk_model_random_forest_validation_SYNCED.ipynb`.


## 1. Setup, Imports, and Configuration

Is cell mein libraries, random seed, feature lists, aur dataset path define hotay hain. Notebook standalone hai: agar file ka naam `(1)` ke saath ho to bhi automatically find ho jaye ga.


In [ ]:
from __future__ import annotations

import json
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path

# Thread limits make sklearn execution stable in small notebook environments.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
    silhouette_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
N_CLUSTERS = 3
CV_STRATIFY_BINS = 10
N_BOOTSTRAP = 300
FINAL_MODEL_NAME = "Multilayer Perceptron"

NOTEBOOK_CWD = Path.cwd().resolve()
ROOT = next(
    (p for p in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (p / "data").exists()),
    NOTEBOOK_CWD,
)
ARTIFACTS_DIR = ROOT / "results" / "reproduced_artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_CANDIDATES = [
    ROOT / "data" / "raw" / "personal_finance_tracker_dataset.csv",
    ROOT / "data" / "sample_inputs" / "personal_finance_tracker_dataset.csv",
    ROOT / "data" / "personal_finance_tracker_dataset.csv",
]
RAW_DATASET_PATH = next((p for p in DATASET_CANDIDATES if p.exists()), None)
if RAW_DATASET_PATH is None:
    raise FileNotFoundError("Dataset CSV not found. Put personal_finance_tracker_dataset.csv in the notebook folder.")

EXPENSE_COMPONENT_COLUMNS = ["rent_or_mortgage", "essential_spending", "discretionary_spending"]
CLUSTER_FEATURES = ["surplus", "expense_ratio", "debt_pressure"]
CLASSIFICATION_FEATURES = [
    "monthly_income",
    "credit_score",
    "rent_or_mortgage",
    "essential_spending",
    "discretionary_spending",
]
LEAKY_COLUMNS = [
    "user_id",
    "date",
    "cash_flow_status",
    "financial_advice_score",
    "financial_stress_level",
    "actual_savings",
    "savings_goal_met",
]
BLOCKED_CLASSIFIER_COLUMNS = [
    *LEAKY_COLUMNS,
    "monthly_expense_total",
    "surplus",
    "expense_ratio",
    "debt_pressure",
    "cluster_id",
    "cluster",
    "risk_level",
    "risk_band",
]
RISK_BANDS = {0: "Low Risk", 1: "Medium Risk", 2: "High Risk"}
ORDERED_LABELS = sorted(RISK_BANDS)
ORDERED_NAMES = [RISK_BANDS[i] for i in ORDERED_LABELS]

print("Dataset path:", RAW_DATASET_PATH)
print("sklearn version:", sklearn.__version__)
print("Final production model:", FINAL_MODEL_NAME)
print("Classifier features:", CLASSIFICATION_FEATURES)


## 2. Load Data and Correct Financial Features

Monthly expenses ko raw `monthly_expense_total` par blindly rely nahi kiya gaya. Is notebook mein expenses dubara calculate hotay hain:

`monthly_expense_total = rent_or_mortgage + essential_spending + discretionary_spending`

Phir `surplus`, `expense_ratio`, aur `debt_pressure` banaye jatay hain.


In [ ]:
def load_raw_data(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)


def correct_financial_features(df: pd.DataFrame, copy: bool = True) -> pd.DataFrame:
    out = df.copy() if copy else df
    required = ["monthly_income", "loan_payment", *EXPENSE_COMPONENT_COLUMNS]
    missing = [col for col in required if col not in out.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    out["monthly_expense_total"] = out[EXPENSE_COMPONENT_COLUMNS].sum(axis=1)
    out["surplus"] = out["monthly_income"] - out["monthly_expense_total"]

    safe_income = out["monthly_income"].replace(0, np.nan)
    out["expense_ratio"] = (out["monthly_expense_total"] / safe_income) * 100
    out["debt_pressure"] = (out["loan_payment"] / safe_income) * 100

    # Zero-income / invalid denominator sentinel for model safety.
    out[["expense_ratio", "debt_pressure"]] = (
        out[["expense_ratio", "debt_pressure"]]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(999)
    )
    return out


raw_df = correct_financial_features(load_raw_data(RAW_DATASET_PATH))
print(f"Shape: {raw_df.shape}")
print("Columns:", raw_df.columns.tolist())
display(raw_df.head())


## 3. Feature Contract and Leakage Validation

Yahan check hota hai ke classifier mein target-like/leaky columns include na hon. Risk labels clustering se banay jatay hain, is liye supervised models ko direct outcome columns nahi diye jatay.


In [ ]:
def prepare_cluster_features(df: pd.DataFrame) -> pd.DataFrame:
    return df[CLUSTER_FEATURES].copy()


def prepare_classification_features(df: pd.DataFrame) -> pd.DataFrame:
    return df[CLASSIFICATION_FEATURES].copy()


def validate_feature_contracts(df: pd.DataFrame, cluster_features: pd.DataFrame, classification_features: pd.DataFrame) -> None:
    missing_cluster = [col for col in CLUSTER_FEATURES if col not in cluster_features.columns]
    missing_classifier = [col for col in CLASSIFICATION_FEATURES if col not in classification_features.columns]
    if missing_cluster or missing_classifier:
        raise ValueError(f"Missing features. cluster={missing_cluster}, classifier={missing_classifier}")

    blocked_overlap = sorted(set(classification_features.columns).intersection(BLOCKED_CLASSIFIER_COLUMNS))
    if blocked_overlap:
        raise ValueError(f"Classifier features contain blocked/leakage columns: {blocked_overlap}")

    numeric_check = pd.concat([cluster_features, classification_features], axis=1)
    if not np.isfinite(numeric_check.to_numpy(dtype=float)).all():
        raise ValueError("Non-finite values found after feature engineering.")

    print("Feature validation passed.")
    print("Cluster features:", CLUSTER_FEATURES)
    print("Classifier features:", CLASSIFICATION_FEATURES)
    print("Blocked classifier columns:", BLOCKED_CLASSIFIER_COLUMNS)


cluster_features = prepare_cluster_features(raw_df)
classification_features = prepare_classification_features(raw_df)
validate_feature_contracts(raw_df, cluster_features, classification_features)

display(classification_features.describe().T)


## 4. EDA — Correlation Heatmap

Correlation plot se feature relationship samajh aati hai. High correlation ko note karna zaroori hai, lekin final validation holdout/CV par ki gayi hai.


In [ ]:
candidate_numeric = [
    "monthly_income",
    "credit_score",
    "rent_or_mortgage",
    "essential_spending",
    "discretionary_spending",
    "monthly_expense_total",
    "loan_payment",
    "surplus",
    "expense_ratio",
    "debt_pressure",
]
available_numeric = [col for col in candidate_numeric if col in raw_df.columns]
correlation_table = raw_df[available_numeric].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(correlation_table, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap for finance features")
plt.tight_layout()
plt.show()


## 5. KMeans Risk Labeling Logic

Risk labels direct target columns se nahi liye gaye. Pehle KMeans `surplus`, `expense_ratio`, aur `debt_pressure` par fit hota hai. Phir clusters ko severity score ke basis par map kiya jata hai:

`severity_score = expense_ratio + debt_pressure - normalized_surplus_signal`

Lowest severity = Low Risk, highest severity = High Risk.


In [ ]:
def fit_clusterer(cluster_frame: pd.DataFrame) -> dict:
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    matrix = imputer.fit_transform(cluster_frame[CLUSTER_FEATURES])
    matrix = scaler.fit_transform(matrix)

    model = KMeans(n_clusters=N_CLUSTERS, n_init=50, random_state=RANDOM_STATE)
    labels = model.fit_predict(matrix)

    sil = float("nan")
    if len(np.unique(labels)) > 1:
        sil = float(silhouette_score(matrix, labels))

    return {"imputer": imputer, "scaler": scaler, "model": model, "labels": labels, "silhouette": sil, "k": N_CLUSTERS}


def predict_clusters(clusterer: dict, cluster_frame: pd.DataFrame) -> np.ndarray:
    matrix = clusterer["imputer"].transform(cluster_frame[CLUSTER_FEATURES])
    matrix = clusterer["scaler"].transform(matrix)
    return clusterer["model"].predict(matrix)


def build_risk_map(train_cluster_features: pd.DataFrame, cluster_labels: np.ndarray) -> tuple[dict[int, int], pd.DataFrame]:
    profile = train_cluster_features.copy()
    profile["cluster_id"] = cluster_labels
    profile = profile.groupby("cluster_id", as_index=True).mean()
    profile["cluster_size"] = pd.Series(cluster_labels).value_counts().sort_index()

    surplus_component = -profile["surplus"].rank(method="dense", ascending=True)
    profile["severity_score"] = profile["expense_ratio"] + profile["debt_pressure"] + surplus_component

    ordered_clusters = profile["severity_score"].sort_values().index.tolist()
    risk_map = {int(cluster_id): int(risk_level) for risk_level, cluster_id in enumerate(ordered_clusters)}
    profile["risk_level"] = profile.index.map(risk_map)
    profile["risk_band"] = profile["risk_level"].map(RISK_BANDS)
    return risk_map, profile.sort_values("risk_level")


def build_risk_labels(cluster_labels: np.ndarray, risk_map: dict[int, int], split_name: str = "dataset") -> pd.Series:
    missing = sorted(set(map(int, np.unique(cluster_labels))) - set(risk_map.keys()))
    if missing:
        raise ValueError(f"Missing risk mapping for {split_name}: {missing}")
    return pd.Series(cluster_labels).map(risk_map).astype(int)


full_clusterer_preview = fit_clusterer(cluster_features)
preview_risk_map, preview_cluster_profile = build_risk_map(cluster_features, full_clusterer_preview["labels"])
print(f"Full-data KMeans silhouette preview: {full_clusterer_preview['silhouette']:.4f}")
display(preview_cluster_profile)


## 5A. Clustering Separation Visualization

Yeh visualization KMeans clusters ki separation ko do angles se show karta hai:

- **PCA 2D view:** scaled clustering features ko 2 dimensions mein project karta hai aur cluster centroids mark karta hai.
- **Financial feature view:** expense ratio vs surplus plot karta hai, jahan point size debt pressure ko show karta hai.


In [ ]:
from sklearn.decomposition import PCA

def plot_cluster_separation(cluster_features: pd.DataFrame, clusterer: dict, risk_map: dict[int, int]) -> pd.DataFrame:
    matrix = clusterer["imputer"].transform(cluster_features[CLUSTER_FEATURES])
    matrix = clusterer["scaler"].transform(matrix)
    labels = clusterer["labels"].astype(int)

    pca = PCA(n_components=2)
    projected = pca.fit_transform(matrix)
    centers_projected = pca.transform(clusterer["model"].cluster_centers_)

    plot_df = cluster_features.copy()
    plot_df["cluster_id"] = labels
    plot_df["risk_level"] = plot_df["cluster_id"].map(risk_map).astype(int)
    plot_df["risk_band"] = plot_df["risk_level"].map(RISK_BANDS)
    plot_df["pca_1"] = projected[:, 0]
    plot_df["pca_2"] = projected[:, 1]
    plot_df["expense_ratio_plot"] = plot_df["expense_ratio"].clip(upper=150)
    surplus_bounds = plot_df["surplus"].quantile([0.01, 0.99])
    plot_df["surplus_plot"] = plot_df["surplus"].clip(surplus_bounds.iloc[0], surplus_bounds.iloc[1])
    plot_df["debt_pressure_plot"] = plot_df["debt_pressure"].clip(upper=60)

    center_df = pd.DataFrame(
        {
            "cluster_id": np.arange(clusterer["k"]),
            "pca_1": centers_projected[:, 0],
            "pca_2": centers_projected[:, 1],
        }
    )
    center_df["risk_level"] = center_df["cluster_id"].map(risk_map).astype(int)
    center_df["risk_band"] = center_df["risk_level"].map(RISK_BANDS)

    palette = {"Low Risk": "#059669", "Medium Risk": "#d97706", "High Risk": "#e11d48"}
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

    sns.scatterplot(
        data=plot_df,
        x="pca_1",
        y="pca_2",
        hue="risk_band",
        style="cluster_id",
        palette=palette,
        s=58,
        alpha=0.72,
        edgecolor="white",
        linewidth=0.35,
        ax=axes[0],
    )
    for row in center_df.itertuples(index=False):
        axes[0].scatter(
            row.pca_1,
            row.pca_2,
            marker="X",
            s=260,
            color=palette[row.risk_band],
            edgecolor="#0b1f3a",
            linewidth=1.4,
            zorder=5,
        )
        axes[0].text(
            row.pca_1,
            row.pca_2,
            f" C{row.cluster_id} / {row.risk_band.replace(' Risk', '')}",
            color="#0b1f3a",
            weight="bold",
            fontsize=9,
            va="center",
        )
    axes[0].set_title(
        f"KMeans separation in PCA space | silhouette={clusterer['silhouette']:.3f}",
        weight="bold",
        color="#0b1f3a",
    )
    axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)")
    axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)")
    axes[0].grid(alpha=0.20)

    sns.scatterplot(
        data=plot_df,
        x="expense_ratio_plot",
        y="surplus_plot",
        hue="risk_band",
        size="debt_pressure_plot",
        sizes=(35, 210),
        palette=palette,
        alpha=0.70,
        edgecolor="white",
        linewidth=0.35,
        ax=axes[1],
    )
    axes[1].axhline(0, color="#0b1f3a", linestyle="--", linewidth=1.1, alpha=0.72)
    axes[1].axvline(60, color="#d97706", linestyle="--", linewidth=1.1, alpha=0.72)
    axes[1].axvline(90, color="#e11d48", linestyle="--", linewidth=1.1, alpha=0.72)
    axes[1].set_title("Financial feature separation", weight="bold", color="#0b1f3a")
    axes[1].set_xlabel("Expense ratio (%) capped at 150")
    axes[1].set_ylabel("Monthly surplus clipped to 1st-99th percentile")
    axes[1].grid(alpha=0.20)

    for ax in axes:
        legend = ax.legend(frameon=True, facecolor="white", edgecolor="#c8d9ef", fontsize=9)
        if legend is not None:
            legend.get_frame().set_alpha(0.92)

    fig.suptitle("KMeans Risk Cluster Separation", fontsize=15, weight="bold", color="#0b1f3a")
    fig.savefig(ARTIFACTS_DIR / "cluster_separation_visualization.png", dpi=180, bbox_inches="tight")
    plt.show()
    return plot_df

cluster_separation_df = plot_cluster_separation(cluster_features, full_clusterer_preview, preview_risk_map)
display(
    cluster_separation_df.groupby(["cluster_id", "risk_band"])[CLUSTER_FEATURES]
    .mean()
    .round(2)
)


## 6. Production-Aligned Holdout Split and Candidate Models

Same production-style train/test split aur KMeans-derived risk labels par chaar models compare honge:

- Logistic Regression
- Decision Tree
- Multilayer Perceptron
- Random Forest

Important: split `budget_goal` se nahi hota; yahan production ke mutabiq `monthly_income` quantile bins use hotay hain.


In [ ]:
def stable_quantile_bins(series: pd.Series, q: int = CV_STRATIFY_BINS) -> pd.Series:
    ranked = series.rank(method="first")
    return pd.qcut(ranked, q=q, labels=False, duplicates="drop")


def build_preprocessor(features: pd.DataFrame) -> ColumnTransformer:
    numeric_cols = features.columns.tolist()
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            )
        ]
    )


def build_model_pipeline(features: pd.DataFrame, estimator) -> Pipeline:
    return Pipeline(steps=[("preprocessor", build_preprocessor(features)), ("model", estimator)])


def get_candidate_estimators() -> dict:
    return {
        "Logistic Regression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "Decision Tree": DecisionTreeClassifier(
            max_depth=6,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "Multilayer Perceptron": MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=0.001,
            learning_rate_init=0.001,
            max_iter=1000,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=20,
            random_state=RANDOM_STATE,
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=1,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }


def build_holdout_dataset(raw_df: pd.DataFrame, cluster_features: pd.DataFrame, classification_features: pd.DataFrame) -> dict:
    # Production split: stratify by deployable monthly income, not budget_goal or target-like columns.
    stratify_target = stable_quantile_bins(classification_features["monthly_income"])
    train_idx, test_idx = train_test_split(
        np.arange(len(classification_features)),
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=stratify_target,
    )

    x_train = classification_features.iloc[train_idx].reset_index(drop=True)
    x_test = classification_features.iloc[test_idx].reset_index(drop=True)
    train_cluster = cluster_features.iloc[train_idx].reset_index(drop=True)
    test_cluster = cluster_features.iloc[test_idx].reset_index(drop=True)

    clusterer = fit_clusterer(train_cluster)
    train_clusters = clusterer["labels"]
    test_clusters = predict_clusters(clusterer, test_cluster)
    risk_map, cluster_profile = build_risk_map(train_cluster, train_clusters)

    y_train = build_risk_labels(train_clusters, risk_map, split_name="holdout train")
    y_test = build_risk_labels(test_clusters, risk_map, split_name="holdout test")

    return {
        "train_idx": train_idx,
        "test_idx": test_idx,
        "x_train": x_train,
        "x_test": x_test,
        "y_train": y_train,
        "y_test": y_test,
        "clusterer": clusterer,
        "risk_map": risk_map,
        "cluster_profile": cluster_profile,
    }


holdout_data = build_holdout_dataset(raw_df, cluster_features, classification_features)
print("Holdout label distribution:")
display(holdout_data["y_test"].map(RISK_BANDS).value_counts().rename("count").to_frame())
display(holdout_data["cluster_profile"])


## 7. Production-Aligned Model Comparison on Holdout Data

Yahan accuracy, macro F1, ROC-AUC, aur PR-AUC compare hotay hain. Notebook comparison production pipeline ke same feature contract use karta hai aur MLP ko same scaled numeric features par evaluate karta hai. Final selected production model ab `Multilayer Perceptron` hai, and the table shows each candidate's holdout performance side by side.


In [ ]:
def safe_predict_proba(model: Pipeline, x: pd.DataFrame) -> np.ndarray:
    if not hasattr(model, "predict_proba"):
        raise AttributeError("Model does not support predict_proba")
    return model.predict_proba(x)


def evaluate_candidate_models(holdout_data: dict) -> tuple[pd.DataFrame, dict]:
    trained_models = {}
    rows = []
    y_train = holdout_data["y_train"]
    y_test = holdout_data["y_test"]
    y_test_bin = label_binarize(y_test, classes=ORDERED_LABELS)

    for model_name, estimator in get_candidate_estimators().items():
        model = build_model_pipeline(holdout_data["x_train"], estimator)
        model.fit(holdout_data["x_train"], y_train)
        trained_models[model_name] = model

        train_pred = model.predict(holdout_data["x_train"])
        test_pred = model.predict(holdout_data["x_test"])
        test_proba = safe_predict_proba(model, holdout_data["x_test"])

        rows.append(
            {
                "model": model_name,
                "train_accuracy": accuracy_score(y_train, train_pred),
                "test_accuracy": accuracy_score(y_test, test_pred),
                "train_macro_f1": f1_score(y_train, train_pred, average="macro", zero_division=0),
                "test_macro_f1": f1_score(y_test, test_pred, average="macro", zero_division=0),
                "roc_auc_ovr_macro": roc_auc_score(y_test, test_proba, multi_class="ovr", average="macro"),
                "pr_auc_macro": average_precision_score(y_test_bin, test_proba, average="macro"),
            }
        )

    comparison_table = (
        pd.DataFrame(rows)
        .sort_values(["test_macro_f1", "test_accuracy"], ascending=False)
        .reset_index(drop=True)
    )
    return comparison_table, trained_models


model_comparison_table, trained_holdout_models = evaluate_candidate_models(holdout_data)
SELECTED_MODEL_NAME = FINAL_MODEL_NAME

print("Production-aligned model comparison - sorted by macro F1 then accuracy:")
display(model_comparison_table.style.format({
    "train_accuracy": "{:.4f}",
    "test_accuracy": "{:.4f}",
    "train_macro_f1": "{:.4f}",
    "test_macro_f1": "{:.4f}",
    "roc_auc_ovr_macro": "{:.4f}",
    "pr_auc_macro": "{:.4f}",
}))
print(f"Selected final production model: {SELECTED_MODEL_NAME}")

selected_row_for_diagnosis = model_comparison_table.loc[model_comparison_table["model"] == FINAL_MODEL_NAME].iloc[0]
comparison_diagnosis_rows = []
for other_model_name in model_comparison_table["model"]:
    if other_model_name == FINAL_MODEL_NAME:
        continue
    other_row = model_comparison_table.loc[model_comparison_table["model"] == other_model_name].iloc[0]
    comparison_diagnosis_rows.append(
        {
            "check": f"{FINAL_MODEL_NAME} minus {other_model_name}",
            "accuracy_delta": selected_row_for_diagnosis["test_accuracy"] - other_row["test_accuracy"],
            "macro_f1_delta": selected_row_for_diagnosis["test_macro_f1"] - other_row["test_macro_f1"],
            "interpretation": f"positive means {FINAL_MODEL_NAME} is better on the production holdout",
        }
    )
comparison_diagnosis = pd.DataFrame(comparison_diagnosis_rows)
display(comparison_diagnosis.style.format({"accuracy_delta": "{:+.4f}", "macro_f1_delta": "{:+.4f}"}))


## 7A. Reusable Evaluation Plotting Helpers

These helper functions reuse the existing holdout split, trained candidate pipelines, and risk-label mapping. They only create evaluation visualizations and do not alter the production model bundle, feature contract, recommendation logic, or saved artifact schema.


In [ ]:
from sklearn.metrics import auc, log_loss, precision_score, recall_score


def _get_model_classes(model) -> np.ndarray:
    classes = getattr(model, "classes_", None)
    if classes is None and hasattr(model, "named_steps"):
        classes = getattr(model.named_steps.get("model"), "classes_", None)
    if classes is None:
        raise AttributeError("Could not resolve class order from the trained model.")
    return np.asarray(classes)


def _aligned_predict_proba(model, x: pd.DataFrame, ordered_labels: list[int] | None = None) -> np.ndarray:
    ordered_labels = ORDERED_LABELS if ordered_labels is None else ordered_labels
    proba = safe_predict_proba(model, x)
    model_classes = _get_model_classes(model)
    aligned = np.zeros((len(x), len(ordered_labels)), dtype=float)
    label_positions = {int(label): pos for pos, label in enumerate(ordered_labels)}

    for class_pos, class_label in enumerate(model_classes):
        class_label = int(class_label)
        if class_label in label_positions:
            aligned[:, label_positions[class_label]] = proba[:, class_pos]
    return aligned


def _final_mlp_holdout_model():
    if SELECTED_MODEL_NAME not in trained_holdout_models:
        raise KeyError(f"{SELECTED_MODEL_NAME!r} was not found in trained_holdout_models.")
    return trained_holdout_models[SELECTED_MODEL_NAME]


def plot_model_comparison(trained_models: dict, x_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame:
    model_order = [
        "Logistic Regression",
        "Decision Tree",
        "Multilayer Perceptron",
        "Random Forest",
    ]
    rows = []
    for model_name in model_order:
        if model_name not in trained_models:
            continue
        model = trained_models[model_name]
        pred = model.predict(x_test)
        rows.append({
            "Models": model_name,
            "Precision": precision_score(y_test, pred, labels=ORDERED_LABELS, average="weighted", zero_division=0),
            "Recall": recall_score(y_test, pred, labels=ORDERED_LABELS, average="weighted", zero_division=0),
            "F1-score": f1_score(y_test, pred, labels=ORDERED_LABELS, average="weighted", zero_division=0),
            "Accuracy": accuracy_score(y_test, pred),
        })

    comparison_df = pd.DataFrame(rows)
    metrics = ["Precision", "Recall", "F1-score", "Accuracy"]
    x_positions = np.arange(len(comparison_df))
    width = 0.18

    fig, ax = plt.subplots(figsize=(11, 6))
    for metric_idx, metric in enumerate(metrics):
        offsets = x_positions + (metric_idx - (len(metrics) - 1) / 2) * width
        values = comparison_df[metric].to_numpy(dtype=float)
        ax.bar(offsets, values, width=width, label=metric)
        best_score = values.max()
        for x_pos, value in zip(offsets, values):
            if np.isclose(value, best_score):
                ax.annotate(
                    "Best",
                    xy=(x_pos, value),
                    xytext=(x_pos, min(1.11, value + 0.08)),
                    ha="center",
                    va="bottom",
                    fontsize=9,
                    arrowprops={"arrowstyle": "->", "lw": 0.9, "color": "#333333"},
                )

    ax.set_title("Performance Comparison", fontsize=14, fontweight="bold")
    ax.set_xlabel("Models")
    ax.set_ylabel("Score")
    ax.set_xticks(x_positions)
    ax.set_xticklabels(comparison_df["Models"], rotation=15, ha="right")
    ax.set_ylim(0, 1.15)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
    return comparison_df


def plot_multiclass_roc(model, x_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame:
    y_test_bin = label_binarize(y_test, classes=ORDERED_LABELS)
    proba = _aligned_predict_proba(model, x_test)
    rows = []

    fig, ax = plt.subplots(figsize=(8, 6))
    for class_idx, class_label in enumerate(ORDERED_LABELS):
        if np.unique(y_test_bin[:, class_idx]).size < 2:
            continue
        fpr, tpr, _ = roc_curve(y_test_bin[:, class_idx], proba[:, class_idx])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2, label=f"{RISK_BANDS[class_label]} (AUC = {roc_auc:.3f})")
        rows.append({"risk_label": RISK_BANDS[class_label], "auc": roc_auc})

    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.5, label="Random baseline")
    ax.set_title("Receiver Operating Characteristic (ROC) Curve", fontsize=14, fontweight="bold")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(rows)


def plot_multiclass_pr_curve(model, x_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame:
    y_test_bin = label_binarize(y_test, classes=ORDERED_LABELS)
    proba = _aligned_predict_proba(model, x_test)
    rows = []

    fig, ax = plt.subplots(figsize=(8, 6))
    for class_idx, class_label in enumerate(ORDERED_LABELS):
        if np.unique(y_test_bin[:, class_idx]).size < 2:
            continue
        precision, recall, _ = precision_recall_curve(y_test_bin[:, class_idx], proba[:, class_idx])
        ap_score = average_precision_score(y_test_bin[:, class_idx], proba[:, class_idx])
        ax.plot(recall, precision, linewidth=2, label=f"{RISK_BANDS[class_label]} (AP = {ap_score:.3f})")
        rows.append({"risk_label": RISK_BANDS[class_label], "average_precision": ap_score})

    ax.set_title("Precision-Recall Curve", fontsize=14, fontweight="bold")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)
    ax.legend(loc="lower left")
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(rows)


def plot_uncertainty_accuracy_curve(model, x_test: pd.DataFrame, y_test: pd.Series, points: int = 20) -> pd.DataFrame:
    proba = _aligned_predict_proba(model, x_test)
    ordered_label_array = np.asarray(ORDERED_LABELS)
    pred = ordered_label_array[np.argmax(proba, axis=1)]
    confidence = np.max(proba, axis=1)
    y_true = np.asarray(y_test)

    sort_order = np.argsort(-confidence)
    retention_rates = np.linspace(0.05, 1.0, points)
    rows = []
    for retention_rate in retention_rates:
        keep_count = max(1, int(np.ceil(retention_rate * len(y_true))))
        kept_idx = sort_order[:keep_count]
        rows.append({
            "Retention Rate": keep_count / len(y_true),
            "Accuracy": accuracy_score(y_true[kept_idx], pred[kept_idx]),
            "Samples Kept": keep_count,
        })

    uncertainty_df = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(8, 5.5))
    ax.plot(
        uncertainty_df["Retention Rate"],
        uncertainty_df["Accuracy"],
        marker="o",
        linewidth=2,
        label="Accuracy on retained samples",
    )
    ax.set_title("Uncertainty-Accuracy Curve", fontsize=14, fontweight="bold")
    ax.set_xlabel("Retention Rate (Fraction of Confident Samples Kept)")
    ax.set_ylabel("Accuracy")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
    return uncertainty_df


def plot_training_history_or_mlp_learning_curve(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_validation: pd.DataFrame,
    y_validation: pd.Series,
    max_epochs: int = 100,
) -> pd.DataFrame:
    base_estimator = clone(get_candidate_estimators()[SELECTED_MODEL_NAME])
    if not isinstance(base_estimator, MLPClassifier):
        raise TypeError("The selected final estimator must be an sklearn MLPClassifier for this training-history plot.")

    preprocessor = build_preprocessor(x_train)
    x_train_processed = preprocessor.fit_transform(x_train)
    x_validation_processed = preprocessor.transform(x_validation)
    classes = np.asarray(ORDERED_LABELS)

    mlp = clone(base_estimator).set_params(
        early_stopping=False,
        warm_start=True,
        max_iter=1,
        n_iter_no_change=max_epochs + 1,
    )

    y_train_array = np.asarray(y_train)
    y_validation_array = np.asarray(y_validation)
    history = []

    for epoch in range(1, max_epochs + 1):
        mlp.partial_fit(x_train_processed, y_train_array, classes=classes)
        train_pred = mlp.predict(x_train_processed)
        validation_pred = mlp.predict(x_validation_processed)
        train_proba = mlp.predict_proba(x_train_processed)
        validation_proba = mlp.predict_proba(x_validation_processed)

        history.append({
            "epoch": epoch,
            "training_accuracy": accuracy_score(y_train_array, train_pred),
            "validation_accuracy": accuracy_score(y_validation_array, validation_pred),
            "training_loss": log_loss(y_train_array, train_proba, labels=classes),
            "validation_loss": log_loss(y_validation_array, validation_proba, labels=classes),
        })

    history_df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("MLP Training and Validation Performance", fontsize=15, fontweight="bold")

    axes[0].plot(history_df["epoch"], history_df["training_accuracy"], linewidth=2, label="Training Accuracy")
    axes[0].plot(history_df["epoch"], history_df["validation_accuracy"], linewidth=2, label="Validation Accuracy")
    axes[0].set_title("Accuracy Curve")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(alpha=0.3)
    axes[0].legend(loc="lower right")

    axes[1].plot(history_df["epoch"], history_df["training_loss"], linewidth=2, label="Training Loss")
    axes[1].plot(history_df["epoch"], history_df["validation_loss"], linewidth=2, label="Validation Loss")
    axes[1].set_title("Loss Curve")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Log Loss")
    axes[1].grid(alpha=0.3)
    axes[1].legend(loc="upper right")

    plt.tight_layout(rect=[0, 0.02, 1, 0.93])
    plt.show()
    return history_df


## 7B. Model Performance Comparison Bar Chart

This grouped bar chart compares the candidate models using weighted Precision, Recall, F1-score, and Accuracy on the existing holdout test set. Weighted averages are used because the target is multi-class, and the annotation highlights the best model for each metric without changing the selected production model.


In [ ]:
model_comparison_bar_metrics = plot_model_comparison(
    trained_holdout_models,
    holdout_data["x_test"],
    holdout_data["y_test"],
)
display(model_comparison_bar_metrics.style.format({
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1-score": "{:.4f}",
    "Accuracy": "{:.4f}",
}))


## 8. ROC and Precision-Recall Curves for All Candidate Models

Neeche har model ke liye screenshot jaisi ROC aur Precision-Recall curves draw hoti hain. Multi-class target ke liye yahan **micro-average** curve use ki gayi hai, jo overall class separation performance show karti hai.


In [ ]:
def plot_model_curves(trained_models: dict, x_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame:
    y_test_bin = label_binarize(y_test, classes=ORDERED_LABELS)
    curve_rows = []

    n_models = len(trained_models)
    fig, axes = plt.subplots(n_models, 2, figsize=(14, 5 * n_models))
    if n_models == 1:
        axes = np.array([axes])

    for row_idx, (model_name, model) in enumerate(trained_models.items()):
        proba = safe_predict_proba(model, x_test)

        fpr, tpr, _ = roc_curve(y_test_bin.ravel(), proba.ravel())
        roc_auc = roc_auc_score(y_test_bin, proba, average="micro", multi_class="ovr")

        precision, recall, _ = precision_recall_curve(y_test_bin.ravel(), proba.ravel())
        pr_auc = average_precision_score(y_test_bin, proba, average="micro")

        ax_roc = axes[row_idx, 0]
        ax_pr = axes[row_idx, 1]

        ax_roc.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
        ax_roc.plot([0, 1], [0, 1], "--", color="gray")
        ax_roc.set_title(f"{model_name} ROC Curve")
        ax_roc.set_xlabel("False Positive Rate")
        ax_roc.set_ylabel("True Positive Rate")
        ax_roc.set_xlim(0, 1)
        ax_roc.set_ylim(0, 1.05)
        ax_roc.legend(loc="lower right")

        ax_pr.plot(recall, precision, label=f"PR-AUC = {pr_auc:.4f}")
        ax_pr.set_title(f"{model_name} Precision-Recall Curve")
        ax_pr.set_xlabel("Recall")
        ax_pr.set_ylabel("Precision")
        ax_pr.set_xlim(0, 1)
        ax_pr.set_ylim(0, 1.05)
        ax_pr.legend(loc="lower left")

        curve_rows.append({"model": model_name, "micro_roc_auc": roc_auc, "micro_pr_auc": pr_auc})

    plt.tight_layout()
    plt.show()
    return pd.DataFrame(curve_rows).sort_values("micro_roc_auc", ascending=False).reset_index(drop=True)


curve_auc_table = plot_model_curves(trained_holdout_models, holdout_data["x_test"], holdout_data["y_test"])
display(curve_auc_table.style.format({"micro_roc_auc": "{:.4f}", "micro_pr_auc": "{:.4f}"}))


## 9. Statistical Validation

Is section mein model comparison ko statistically validate kiya gaya hai:

- 5-fold cross-validation mean/std
- Holdout bootstrap confidence intervals
- McNemar test style pairwise disagreement check selected model vs baqi models


In [ ]:
def evaluate_cv_for_model(estimator, cluster_features: pd.DataFrame, classification_features: pd.DataFrame) -> dict:
    # Production-style outer CV: each fold refits KMeans on train and predicts risk labels on test.
    pseudo_strata = stable_quantile_bins(classification_features["monthly_income"])
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rows = []

    for fold, (train_idx, test_idx) in enumerate(splitter.split(classification_features, pseudo_strata), start=1):
        train_cluster = cluster_features.iloc[train_idx].reset_index(drop=True)
        test_cluster = cluster_features.iloc[test_idx].reset_index(drop=True)
        x_train = classification_features.iloc[train_idx].reset_index(drop=True)
        x_test = classification_features.iloc[test_idx].reset_index(drop=True)

        clusterer = fit_clusterer(train_cluster)
        train_clusters = clusterer["labels"]
        test_clusters = predict_clusters(clusterer, test_cluster)
        risk_map, _ = build_risk_map(train_cluster, train_clusters)
        y_train = build_risk_labels(train_clusters, risk_map, split_name=f"cv fold {fold} train")
        y_test = build_risk_labels(test_clusters, risk_map, split_name=f"cv fold {fold} test")

        model = build_model_pipeline(x_train, clone(estimator))
        model.fit(x_train, y_train)
        pred = model.predict(x_test)
        rows.append(
            {
                "fold": fold,
                "accuracy": accuracy_score(y_test, pred),
                "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
                "silhouette": clusterer["silhouette"],
            }
        )

    fold_table = pd.DataFrame(rows)
    return {
        "cv_accuracy_mean": float(fold_table["accuracy"].mean()),
        "cv_accuracy_std": float(fold_table["accuracy"].std(ddof=0)),
        "cv_macro_f1_mean": float(fold_table["macro_f1"].mean()),
        "cv_macro_f1_std": float(fold_table["macro_f1"].std(ddof=0)),
        "cv_silhouette_mean": float(fold_table["silhouette"].mean()),
    }


def bootstrap_metric_ci(y_true, y_pred, metric_func, n_bootstrap: int = N_BOOTSTRAP, alpha: float = 0.05) -> tuple[float, float]:
    rng = np.random.default_rng(RANDOM_STATE)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    scores = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        scores.append(metric_func(y_true[idx], y_pred[idx]))
    return float(np.quantile(scores, alpha / 2)), float(np.quantile(scores, 1 - alpha / 2))


def mcnemar_chi_square(y_true, pred_a, pred_b) -> dict:
    # Continuity-corrected McNemar statistic. p-value is approximated for df=1 using erfc(sqrt(x/2)).
    import math
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)
    a_correct = pred_a == y_true
    b_correct = pred_b == y_true
    n01 = int(np.sum(a_correct & ~b_correct))
    n10 = int(np.sum(~a_correct & b_correct))
    denom = n01 + n10
    if denom == 0:
        stat = 0.0
        p_value = 1.0
    else:
        stat = (abs(n01 - n10) - 1) ** 2 / denom
        p_value = math.erfc(math.sqrt(stat / 2))
    return {"n_selected_correct_other_wrong": n01, "n_selected_wrong_other_correct": n10, "mcnemar_chi2": stat, "approx_p_value": p_value}


cv_rows = []
for model_name, estimator in get_candidate_estimators().items():
    row = {"model": model_name}
    row.update(evaluate_cv_for_model(estimator, cluster_features, classification_features))
    cv_rows.append(row)
cv_results_table = pd.DataFrame(cv_rows).sort_values("cv_macro_f1_mean", ascending=False).reset_index(drop=True)

selected_model = trained_holdout_models[SELECTED_MODEL_NAME]
selected_pred = selected_model.predict(holdout_data["x_test"])
acc_ci = bootstrap_metric_ci(holdout_data["y_test"], selected_pred, accuracy_score)
f1_ci = bootstrap_metric_ci(
    holdout_data["y_test"],
    selected_pred,
    lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0),
)

mcnemar_rows = []
for other_name, other_model in trained_holdout_models.items():
    if other_name == SELECTED_MODEL_NAME:
        continue
    row = {"selected_model": SELECTED_MODEL_NAME, "other_model": other_name}
    row.update(mcnemar_chi_square(holdout_data["y_test"], selected_pred, other_model.predict(holdout_data["x_test"])))
    mcnemar_rows.append(row)
mcnemar_table = pd.DataFrame(mcnemar_rows)

print("Production-style cross-validation results:")
display(cv_results_table.style.format({
    "cv_accuracy_mean": "{:.4f}",
    "cv_accuracy_std": "{:.4f}",
    "cv_macro_f1_mean": "{:.4f}",
    "cv_macro_f1_std": "{:.4f}",
    "cv_silhouette_mean": "{:.4f}",
}))
print(f"{SELECTED_MODEL_NAME} holdout accuracy 95% bootstrap CI: {acc_ci[0]:.4f} to {acc_ci[1]:.4f}")
print(f"{SELECTED_MODEL_NAME} holdout macro F1 95% bootstrap CI: {f1_ci[0]:.4f} to {f1_ci[1]:.4f}")
print(f"Pairwise McNemar-style checks against {SELECTED_MODEL_NAME}:")
display(mcnemar_table.style.format({"mcnemar_chi2": "{:.4f}", "approx_p_value": "{:.4f}"}))


## 10. Confusion Matrix and Classification Report for Selected Final Model

Is cell se final selected model ka class-wise behavior clear hota hai.


In [ ]:
final_holdout_pred = trained_holdout_models[SELECTED_MODEL_NAME].predict(holdout_data["x_test"])
print(f"Selected model: {SELECTED_MODEL_NAME}")
print(classification_report(
    holdout_data["y_test"],
    final_holdout_pred,
    labels=ORDERED_LABELS,
    target_names=ORDERED_NAMES,
    zero_division=0,
))

cm = confusion_matrix(holdout_data["y_test"], final_holdout_pred, labels=ORDERED_LABELS)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=ORDERED_NAMES)
disp.plot(values_format="d", cmap="Blues")
disp.ax_.grid(False)
plt.title(f"Confusion Matrix — {SELECTED_MODEL_NAME}")
plt.tight_layout()
plt.show()


## 10A. Multi-Class ROC Curve for Final MLP

The final MLP pipeline supports `predict_proba`, so each risk class is evaluated with a one-vs-rest ROC curve. The diagonal line represents random classification, while each legend entry reports the class-specific AUC.


In [ ]:
final_mlp_holdout_model = _final_mlp_holdout_model()
mlp_multiclass_roc_auc = plot_multiclass_roc(
    final_mlp_holdout_model,
    holdout_data["x_test"],
    holdout_data["y_test"],
)
display(mlp_multiclass_roc_auc.style.format({"auc": "{:.4f}"}))


## 10B. Multi-Class Precision-Recall Curve for Final MLP

Precision-Recall curves show how well the final MLP separates each risk class when positive predictions become more or less selective. Average Precision is reported in the legend for Low Risk, Medium Risk, and High Risk.


In [ ]:
mlp_multiclass_pr_auc = plot_multiclass_pr_curve(
    final_mlp_holdout_model,
    holdout_data["x_test"],
    holdout_data["y_test"],
)
display(mlp_multiclass_pr_auc.style.format({"average_precision": "{:.4f}"}))


## 10C. Uncertainty-Accuracy Curve for Final MLP

This curve uses the maximum predicted probability as model confidence. Samples are sorted by confidence, then accuracy is recomputed as progressively less-confident samples are included; a strong deployment model should perform best when only the most confident predictions are retained.


In [ ]:
mlp_uncertainty_accuracy = plot_uncertainty_accuracy_curve(
    final_mlp_holdout_model,
    holdout_data["x_test"],
    holdout_data["y_test"],
)
display(mlp_uncertainty_accuracy.tail().style.format({
    "Retention Rate": "{:.2f}",
    "Accuracy": "{:.4f}",
}))


## 11. Fit Final MLP Bundle on Full Dataset

Ab production-selected MLP full dataset par train hota hai. Aagay ke cells `bundle["classifier"]` use karenge, is liye downstream inference MLP se chalegi.


In [ ]:
def fit_full_bundle(cluster_features: pd.DataFrame, classification_features: pd.DataFrame, selected_model_name: str) -> tuple[dict, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    clusterer = fit_clusterer(cluster_features)
    full_clusters = clusterer["labels"]
    risk_map, cluster_profile = build_risk_map(cluster_features, full_clusters)
    risk_levels = build_risk_labels(full_clusters, risk_map, split_name="full dataset")

    estimator = get_candidate_estimators()[selected_model_name]
    classifier = build_model_pipeline(classification_features, estimator)
    classifier.fit(classification_features, risk_levels)

    labeled_clustered = cluster_features.copy()
    labeled_clustered["cluster_id"] = full_clusters
    labeled_clustered["risk_level"] = risk_levels.values
    labeled_clustered["risk_band"] = labeled_clustered["risk_level"].map(RISK_BANDS)

    labeled_classification = classification_features.copy()
    labeled_classification["risk_level"] = risk_levels.values
    labeled_classification["risk_band"] = labeled_classification["risk_level"].map(RISK_BANDS)

    bundle = {
        "selected_model_name": selected_model_name,
        "final_classifier": classifier.named_steps["model"].__class__.__name__,
        "classifier": classifier,
        "classifier_feature_columns": CLASSIFICATION_FEATURES,
        "blocked_classifier_columns": BLOCKED_CLASSIFIER_COLUMNS,
        "cluster_feature_columns": CLUSTER_FEATURES,
        "risk_bands": RISK_BANDS,
        "removed_leaky_columns": LEAKY_COLUMNS,
        "selected_k": clusterer["k"],
        "cluster_silhouette": clusterer["silhouette"],
        "cluster_imputer": clusterer["imputer"],
        "cluster_scaler": clusterer["scaler"],
        "cluster_model": clusterer["model"],
        "risk_map": risk_map,
    }
    return bundle, cluster_profile, labeled_clustered, labeled_classification


bundle, cluster_profile, labeled_clustered, labeled_classification = fit_full_bundle(
    cluster_features,
    classification_features,
    SELECTED_MODEL_NAME,
)
print(f"Final selected model fitted on full dataset: {bundle['selected_model_name']}")
print(f"Full-data cluster silhouette: {bundle['cluster_silhouette']:.4f}")
display(cluster_profile)


## 12. MLP Training and Validation Performance Curves

This section trains a fresh monitoring-only MLP with the same architecture as the selected final model and tracks performance over epochs using `partial_fit`. The loop uses the existing holdout train/test split for training and validation curves, while the production `bundle["classifier"]` remains untouched.


In [ ]:
mlp_training_history = plot_training_history_or_mlp_learning_curve(
    holdout_data["x_train"],
    holdout_data["y_train"],
    holdout_data["x_test"],
    holdout_data["y_test"],
    max_epochs=100,
)

display(mlp_training_history.tail().style.format({
    "training_accuracy": "{:.4f}",
    "validation_accuracy": "{:.4f}",
    "training_loss": "{:.4f}",
    "validation_loss": "{:.4f}",
}))


## 12B. Final Model Selection Summary

This summary reports the metric winners honestly and then documents the project decision. The deployed final classifier remains the Multi-Layer Perceptron, and the risk-label contract stays fixed as Low Risk, Medium Risk, and High Risk.


In [ ]:
summary_metrics = model_comparison_bar_metrics.copy()
best_accuracy_row = summary_metrics.loc[summary_metrics["Accuracy"].idxmax()]
best_f1_row = summary_metrics.loc[summary_metrics["F1-score"].idxmax()]
selected_summary_row = summary_metrics.loc[summary_metrics["Models"] == SELECTED_MODEL_NAME].iloc[0]

selection_summary = pd.DataFrame([{
    "best_model_by_accuracy": best_accuracy_row["Models"],
    "best_accuracy": best_accuracy_row["Accuracy"],
    "best_model_by_weighted_f1": best_f1_row["Models"],
    "best_weighted_f1": best_f1_row["F1-score"],
    "selected_final_model": SELECTED_MODEL_NAME,
    "selected_model_accuracy": selected_summary_row["Accuracy"],
    "selected_model_weighted_f1": selected_summary_row["F1-score"],
    "risk_labels": ", ".join(ORDERED_NAMES),
}])

display(selection_summary.style.format({
    "best_accuracy": "{:.4f}",
    "best_weighted_f1": "{:.4f}",
    "selected_model_accuracy": "{:.4f}",
    "selected_model_weighted_f1": "{:.4f}",
}))

print(f"Best model by holdout accuracy: {best_accuracy_row['Models']} ({best_accuracy_row['Accuracy']:.4f})")
print(f"Best model by weighted F1-score: {best_f1_row['Models']} ({best_f1_row['F1-score']:.4f})")
print(f"Selected final model for this FYP deployment: {SELECTED_MODEL_NAME}")
print(f"Final risk labels: {', '.join(ORDERED_NAMES)}")

if (best_accuracy_row["Models"] != SELECTED_MODEL_NAME) or (best_f1_row["Models"] != SELECTED_MODEL_NAME):
    print(
        "Note: MLP is retained as the final deployment model based on the project design and selected production pipeline; "
        "the metric winners above are reported without adjustment."
    )


## 13. Inference Helper — Final Risk Prediction

Yeh cell final selected model, cluster risk, aur rule-based guardrails combine karta hai. Final risk kabhi guardrail se kam nahi hota.


In [ ]:
def apply_financial_guardrails(row: pd.Series) -> tuple[int, list[str]]:
    reasons = []
    guardrail_risk = 0
    if row["surplus"] < 0:
        guardrail_risk = max(guardrail_risk, 2)
        reasons.append("negative surplus")
    if row["expense_ratio"] >= 90:
        guardrail_risk = max(guardrail_risk, 2)
        reasons.append("expense ratio >= 90%")
    elif row["expense_ratio"] >= 60:
        guardrail_risk = max(guardrail_risk, 1)
        reasons.append("expense ratio >= 60%")
    if row["debt_pressure"] >= 35:
        guardrail_risk = max(guardrail_risk, 2)
        reasons.append("debt pressure >= 35%")
    elif row["debt_pressure"] >= 20:
        guardrail_risk = max(guardrail_risk, 1)
        reasons.append("debt pressure >= 20%")
    return guardrail_risk, reasons


def infer_cluster_risk(bundle: dict, user_df: pd.DataFrame) -> tuple[int, int, str]:
    user_cluster_features = user_df[bundle["cluster_feature_columns"]]
    transformed = bundle["cluster_scaler"].transform(bundle["cluster_imputer"].transform(user_cluster_features))
    cluster_id = int(bundle["cluster_model"].predict(transformed)[0])
    cluster_risk = int(bundle["risk_map"][cluster_id])
    return cluster_id, cluster_risk, RISK_BANDS[cluster_risk]


def predict_final_risk(user_profile: dict | pd.Series | pd.DataFrame) -> dict:
    if isinstance(user_profile, pd.DataFrame):
        user_df = user_profile.copy()
    else:
        user_df = pd.DataFrame([dict(user_profile)])
    user_df = correct_financial_features(user_df)

    classifier_x = user_df[bundle["classifier_feature_columns"]]
    ml_risk = int(bundle["classifier"].predict(classifier_x)[0])
    ml_proba = bundle["classifier"].predict_proba(classifier_x)[0]
    cluster_id, cluster_risk, cluster_band = infer_cluster_risk(bundle, user_df)
    guardrail_risk, guardrail_reasons = apply_financial_guardrails(user_df.iloc[0])
    final_risk = max(ml_risk, cluster_risk, guardrail_risk)

    return {
        "selected_model": bundle["selected_model_name"],
        "ml_risk": ml_risk,
        "ml_band": RISK_BANDS[ml_risk],
        "ml_probabilities": {RISK_BANDS[i]: float(ml_proba[pos]) for pos, i in enumerate(bundle["classifier"].classes_)},
        "cluster_id": cluster_id,
        "cluster_risk": cluster_risk,
        "cluster_band": cluster_band,
        "guardrail_risk": guardrail_risk,
        "guardrail_band": RISK_BANDS[guardrail_risk],
        "guardrail_reasons": guardrail_reasons,
        "final_risk": final_risk,
        "final_band": RISK_BANDS[final_risk],
    }


sample_user = raw_df.head(1).copy()
final_intelligent_response = predict_final_risk(sample_user)
print(json.dumps(final_intelligent_response, indent=2))


## 14. Save Artifacts

Final selected model bundle, comparison metrics, validation tables, aur labeled datasets save hotay hain.


In [ ]:
def save_outputs() -> None:
    bundle_path = ARTIFACTS_DIR / "finance_risk_bundle_final.joblib"
    metrics_path = ARTIFACTS_DIR / "finance_risk_metrics_final.json"
    comparison_path = ARTIFACTS_DIR / "model_comparison.csv"
    cv_path = ARTIFACTS_DIR / "cv_validation.csv"
    labeled_classification_path = ARTIFACTS_DIR / "classification_dataset_with_risk_labels.csv"
    labeled_clustered_path = ARTIFACTS_DIR / "clustered_dataset_with_risk_labels.csv"

    joblib.dump(bundle, bundle_path)
    model_comparison_table.to_csv(comparison_path, index=False)
    cv_results_table.to_csv(cv_path, index=False)
    labeled_classification.to_csv(labeled_classification_path, index=False)
    labeled_clustered.to_csv(labeled_clustered_path, index=False)

    metrics = {
        "dataset": RAW_DATASET_PATH.name,
        "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "selected_model_name": SELECTED_MODEL_NAME,
        "final_classifier": bundle["final_classifier"],
        "classification_features": CLASSIFICATION_FEATURES,
        "cluster_features": CLUSTER_FEATURES,
        "removed_leaky_columns": LEAKY_COLUMNS,
        "blocked_classifier_columns": BLOCKED_CLASSIFIER_COLUMNS,
        "model_comparison": model_comparison_table.round(6).to_dict(orient="records"),
        "cross_validation": cv_results_table.round(6).to_dict(orient="records"),
        "curve_auc": curve_auc_table.round(6).to_dict(orient="records"),
        "selected_model_accuracy_ci_95": acc_ci,
        "selected_model_macro_f1_ci_95": f1_ci,
        "mcnemar_checks": mcnemar_table.round(6).to_dict(orient="records"),
        "cluster_silhouette_full": bundle["cluster_silhouette"],
        "artifacts": {
            "bundle": str(bundle_path),
            "model_comparison": str(comparison_path),
            "cv_validation": str(cv_path),
            "labeled_classification": str(labeled_classification_path),
            "labeled_clustered": str(labeled_clustered_path),
        },
    }
    metrics_path.write_text(json.dumps(metrics, indent=2, default=str), encoding="utf-8")

    print("Saved artifacts:")
    for path in [bundle_path, metrics_path, comparison_path, cv_path, labeled_classification_path, labeled_clustered_path]:
        print(path)


save_outputs()


## 15. Final Summary

Yeh final quick summary report/presentation mein directly use ho sakti hai.


In [ ]:
print("Clean leakage-safe finance risk pipeline completed")
print(f"Dataset: {RAW_DATASET_PATH.name}")
print(f"Rows: {len(raw_df):,}")
print(f"Cluster features: {CLUSTER_FEATURES}")
print(f"Classification features: {CLASSIFICATION_FEATURES}")
print(f"Selected K: {bundle['selected_k']}")
print(f"Cluster silhouette: {bundle['cluster_silhouette']:.4f}")
print(f"Selected final model: {SELECTED_MODEL_NAME}")
print(f"Best holdout accuracy: {model_comparison_table.iloc[0]['test_accuracy']:.4f}")
print(f"Best holdout macro F1: {model_comparison_table.iloc[0]['test_macro_f1']:.4f}")
print(f"CV accuracy for selected model: {cv_results_table.loc[cv_results_table['model'] == SELECTED_MODEL_NAME, 'cv_accuracy_mean'].iloc[0]:.4f}")
print(f"Final model artifact: {ARTIFACTS_DIR / 'finance_risk_bundle_final.joblib'}")
